# 04 – Wizualizacje do raportu
**Cel:** Przygotowanie finalnych wykresów odpowiadających na wszystkie pytania badawcze. Wykresy zapisywane są do `reports/figures/`.

## 1. Import i wczytanie danych

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 150, 'font.size': 11,
                     'axes.titlesize': 13, 'axes.labelsize': 11})
sns.set_theme(style='whitegrid', palette='muted')
from pathlib import Path
import os
PROJECT_ROOT = Path(os.getcwd())
# Jeśli CWD to notebooks/, cofnij się poziom wyżej
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH    = str(PROJECT_ROOT / 'data' / 'raw' / 'Car_Prices_Poland_Kaggle.csv')
PROC_PATH    = str(PROJECT_ROOT / 'data' / 'processed') + os.sep
FIGURES_PATH = str(PROJECT_ROOT / 'reports' / 'figures') + os.sep
os.makedirs(PROC_PATH, exist_ok=True)
os.makedirs(FIGURES_PATH, exist_ok=True)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('FIGURES_PATH:', FIGURES_PATH)

df = pd.read_csv(PROC_PATH + 'cars_clean.csv')
print(f'Wczytano: {df.shape}')
try:
    results = pd.read_csv(PROC_PATH + 'model_results.csv')
    print('Wczytano wyniki modeli.')
except FileNotFoundError:
    print('Brak model_results.csv – uruchom najpierw notebook 03.')
    results = None

## 2. PB1 – Czynniki wpływające na cenę (korelacje)

In [ ]:
num_cols = [c for c in ['year','mileage','vol_engine','car_age','mileage_per_year','mark_median_price','price'] if c in df.columns]
corr = df[num_cols].corr()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=axes[0])
axes[0].set_title('Macierz korelacji (Pearson)')
price_corr = corr['price'].drop('price').sort_values()
colors = ['tomato' if v < 0 else 'steelblue' for v in price_corr]
axes[1].barh(price_corr.index, price_corr.values, color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Korelacja cech z ceną (Pearson)')
axes[1].set_xlabel('Współczynnik korelacji')
plt.tight_layout()
plt.savefig(FIGURES_PATH + '04_pb1_correlations.png', bbox_inches='tight')
plt.show()

## 3. PB2 – Cena vs. rok produkcji i przebieg

In [ ]:
sample = df.sample(min(8000, len(df)), random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sc1 = axes[0].scatter(sample['year'], sample['price'], alpha=0.15, s=8,
                       c=sample['mileage'], cmap='coolwarm', vmin=0, vmax=300000)
plt.colorbar(sc1, ax=axes[0], label='Przebieg (km)')
axes[0].set_title('Cena vs. rok produkcji\n(kolor = przebieg)')
axes[0].set_xlabel('Rok produkcji')
axes[0].set_ylabel('Cena (PLN)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
sc2 = axes[1].scatter(sample['mileage'], sample['price'], alpha=0.15, s=8,
                       c=sample['year'], cmap='viridis', vmin=2000, vmax=2022)
plt.colorbar(sc2, ax=axes[1], label='Rok produkcji')
axes[1].set_title('Cena vs. przebieg\n(kolor = rok produkcji)')
axes[1].set_xlabel('Przebieg (km)')
axes[1].set_ylabel('Cena (PLN)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig(FIGURES_PATH + '04_pb2_price_year_mileage.png', bbox_inches='tight')
plt.show()

## 4. PB3 – Cena według rodzaju paliwa

In [ ]:
fuel_counts = df['fuel'].value_counts()
valid_fuels = fuel_counts[fuel_counts >= 100].index
df_fuel = df[df['fuel'].isin(valid_fuels)]
order = df_fuel.groupby('fuel')['price'].median().sort_values(ascending=False).index
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.boxplot(data=df_fuel, x='fuel', y='price', order=order,
            palette='Set2', showfliers=False, ax=axes[0])
axes[0].set_title('Rozkład ceny według paliwa\n(bez wartości ekstremalnych)')
axes[0].set_xlabel('Rodzaj paliwa')
axes[0].set_ylabel('Cena (PLN)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].tick_params(axis='x', rotation=30)
medians = df_fuel.groupby('fuel')['price'].median().reindex(order)
axes[1].barh(list(medians.index[::-1]), list(medians.values[::-1]),
             color=sns.color_palette('Set2', len(medians)))
axes[1].set_title('Mediana ceny według paliwa')
axes[1].set_xlabel('Mediana ceny (PLN)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(medians.values[::-1]):
    axes[1].text(v + 200, i, f'{v:,.0f} PLN', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_PATH + '04_pb3_price_by_fuel.png', bbox_inches='tight')
plt.show()

## 5. PB4 – Marki i województwa z najwyższymi cenami

In [ ]:
mark_stats = (df.groupby('mark')
               .agg(median_price=('price','median'), count=('price','count'))
               .query('count >= 50')
               .sort_values('median_price', ascending=False)
               .head(20))
prov_stats = df.groupby('province')['price'].median().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
sns.barplot(x=mark_stats['median_price'], y=mark_stats.index, palette='Blues_d', ax=axes[0])
axes[0].set_title('Top 20 marek – mediana ceny\n(min. 50 ogłoszeń)')
axes[0].set_xlabel('Mediana ceny (PLN)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
sns.barplot(x=prov_stats.values, y=prov_stats.index, palette='Oranges_d', ax=axes[1])
axes[1].set_title('Mediana ceny według województwa')
axes[1].set_xlabel('Mediana ceny (PLN)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig(FIGURES_PATH + '04_pb4_marks_provinces.png', bbox_inches='tight')
plt.show()

## 6. PB5 – Porównanie modeli ML

In [ ]:
if results is not None:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, (metric, ascending) in zip(axes, [('R2', False), ('RMSE', True), ('MAE', True)]):
        sorted_r = results.sort_values(metric, ascending=ascending).reset_index(drop=True)
        palette = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(sorted_r))]
        sns.barplot(x=sorted_r[metric], y=sorted_r['Model'], palette=palette, ax=ax)
        ax.set_title(metric.replace('R2','R²'))
        ax.set_xlabel(metric.replace('R2','R²'))
        if metric == 'R2':
            ax.set_xlim(0, 1)
    plt.suptitle('Porównanie modeli predykcyjnych ceny samochodu', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(FIGURES_PATH + '04_pb5_model_comparison.png', bbox_inches='tight')
    plt.show()
else:
    print('Brak danych – uruchom notebook 03_modeling.ipynb')

## 7. Lista wygenerowanych wykresów

In [ ]:
import os
figures = sorted(f for f in os.listdir(FIGURES_PATH) if f != '.gitkeep')
print(f'Wykresy w {FIGURES_PATH} ({len(figures)} plików):')
for f in figures:
    print(f'  • {f}')